<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Miniprojet_W8_D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

import os
import getpass
from typing import Annotated, Literal, Iterable
from typing_extensions import TypedDict
from random import randint

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.ai import AIMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool

# --- Setup API Key ---
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    except:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Please enter your GOOGLE_API_KEY: ")

# --- Definitions ---
class OrderState(TypedDict):
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool

BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system..."
)
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

# --- Tools ---
@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return "MENU: Espresso, Latte, Cappuccino, Green Tea, Matcha Latte. Modifiers: Oat milk, Almond milk, Extra shot."

@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order."""
    pass

@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct."""
    pass

@tool
def get_order() -> str:
    """Returns the users order so far."""
    pass

@tool
def clear_order():
    """Removes all items from the user's order."""
    pass

@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment."""
    pass

# --- Nodes ---
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")
auto_tools = [get_menu]
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]
llm_with_tools = llm.bind_tools(auto_tools + order_tools)

def chatbot_with_tools(state: OrderState) -> OrderState:
    if not state["messages"]:
        return {"messages": [AIMessage(content=WELCOME_MSG)]}
    return {"messages": [llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])]}

def human_node(state: OrderState) -> OrderState:
    print("Model:", state["messages"][-1].content)
    user_input = input("User: ")
    if user_input.lower() in {"q", "quit", "exit"}:
        return {**state, "finished": True, "messages": [("user", user_input)]}
    return {**state, "messages": [("user", user_input)]}

def order_node(state: OrderState) -> OrderState:
    tool_msg = state["messages"][-1]
    order = state.get("order", []).copy()
    outbound_msgs = []
    order_placed = False
    for tool_call in tool_msg.tool_calls:
        t_name = tool_call["name"]
        if t_name == "add_to_order":
            item = f"{tool_call['args']['drink']} ({', '.join(tool_call['args'].get('modifiers', []))})"
            order.append(item)
            res = "Added."
        elif t_name == "confirm_order":
            res = input(f"Confirming: {order}. Is this correct? ")
        elif t_name == "place_order":
            order_placed = True
            res = f"Done! ETA {randint(1,5)}m."
        else: res = "Done."
        outbound_msgs.append(ToolMessage(content=str(res), name=t_name, tool_call_id=tool_call["id"]))
    return {"messages": outbound_msgs, "order": order, "finished": order_placed}

# --- Routing ---
def maybe_route_to_tools(state: OrderState):
    msg = state["messages"][-1]
    if state.get("finished"): return END
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        if any(tc["name"] in [t.name for t in auto_tools] for tc in msg.tool_calls): return "tools"
        return "ordering"
    return "human"

def maybe_exit_human_node(state: OrderState):
    return END if state.get("finished") else "chatbot"

# --- Final Graph ---
graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", ToolNode(auto_tools))
graph_builder.add_node("ordering", order_node)

graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
graph_builder.add_conditional_edges("human", maybe_exit_human_node)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph_with_order_tools = graph_builder.compile()